# Your Title Here

**Name(s)**: (your name(s) here)

**Website Link**: (your website link)

In [35]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import *

## Step 1: Introduction

In [5]:
path = Path('2025_LoL_esports_match_data_from_OraclesElixir.csv')
lol = pd.read_csv(path)
lol.head()

/var/folders/8z/t3klk_9d3sbf04zdg07sk1ch0000gn/T/ipykernel_68610/1291358343.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  lol = pd.read_csv(path)


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,200.0,224.0,-1.0,17.0,1.0,1.0,2.0,2.0,4.0,2.0
1,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,157.0,-2363.0,-1444.0,-18.0,0.0,1.0,2.0,1.0,7.0,0.0
2,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,241.0,-1552.0,-2465.0,-41.0,1.0,0.0,2.0,1.0,5.0,1.0
3,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,257.0,-2613.0,-1156.0,-6.0,1.0,1.0,2.0,6.0,2.0,0.0
4,LOLTMNT03_179647,complete,NaN,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,...,20.0,-662.0,-734.0,18.0,0.0,2.0,2.0,0.0,8.0,0.0


In [6]:
lol.shape

(120636, 165)

In [7]:
(lol.isna().mean() * 100).sort_values(ascending=False)

dragons (type unknown)     98.645512
url                        91.873073
monsterkillsenemyjungle    91.873073
monsterkillsownjungle      91.873073
atakhans                   84.730926
                             ...    
datacompleteness            0.000000
dpm                         0.000000
damagetochampions           0.000000
ckpm                        0.000000
damagetakenperminute        0.000000
Length: 165, dtype: float64

How does the number of objectives secured affect the outcome of a match?

In [8]:
short = lol[['gameid', 'position', 'side', 'teamname', 'result', 'teamkills', 'teamdeaths', 'dragons', 'elders', 'heralds', 'void_grubs', 'barons', 'atakhans', 'damagetochampions', 'totalgold', 'monsterkills']].copy()
short.head()

,gameid,position,side,teamname,result,teamkills,teamdeaths,dragons,elders,heralds,void_grubs,barons,atakhans,damagetochampions,totalgold,monsterkills
0,LOLTMNT03_179647,top,Blue,IziDream,0,3,13,NaN,NaN,NaN,NaN,0.0,NaN,20156,10668,0
1,LOLTMNT03_179647,jng,Blue,IziDream,0,3,13,NaN,NaN,NaN,NaN,0.0,NaN,4963,7429,132
2,LOLTMNT03_179647,mid,Blue,IziDream,0,3,13,NaN,NaN,NaN,NaN,0.0,NaN,13952,9032,0
3,LOLTMNT03_179647,bot,Blue,IziDream,0,3,13,NaN,NaN,NaN,NaN,0.0,NaN,6898,9407,12
4,LOLTMNT03_179647,sup,Blue,IziDream,0,3,13,NaN,NaN,NaN,NaN,0.0,NaN,4174,5719,0


In [9]:
short = short[short['position'] == 'team'].copy()
short

,gameid,position,side,teamname,result,teamkills,teamdeaths,dragons,elders,heralds,void_grubs,barons,atakhans,damagetochampions,totalgold,monsterkills
10,LOLTMNT03_179647,team,Blue,IziDream,0,3,13,0.0,0.0,0.0,0.0,0.0,0.0,50143,42255,144
11,LOLTMNT03_179647,team,Red,Team Valiant,1,13,3,2.0,0.0,1.0,6.0,1.0,1.0,53681,53936,169
22,LOLTMNT06_96134,team,Blue,Esprit Shōnen,1,21,11,3.0,0.0,1.0,6.0,1.0,1.0,100112,64669,199
23,LOLTMNT06_96134,team,Red,Skillcamp,0,10,21,2.0,0.0,0.0,0.0,0.0,0.0,78010,50679,134
34,LOLTMNT06_95160,team,Blue,Karmine Corp Blue Stars,0,18,22,0.0,0.0,0.0,2.0,0.0,0.0,67870,51389,146
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120611,LOLTMNT03_332178,team,Red,LGD Gaming,1,15,3,3.0,0.0,1.0,3.0,1.0,1.0,58593,60338,193
120622,LOLTMNT03_331511,team,Blue,Anyone's Legend,0,11,24,4.0,0.0,1.0,1.0,0.0,0.0,95153,64282,220
120623,LOLTMNT03_331511,team,Red,LGD Gaming,1,24,11,2.0,0.0,0.0,2.0,2.0,1.0,84235,76363,267
120634,LOLTMNT03_332179,team,Blue,Anyone's Legend,0,9,22,0.0,0.0,1.0,3.0,0.0,0.0,74670,51428,177


In [10]:
obj_cols = ["dragons", "barons", "atakhans", "void_grubs", "elders"]
short["total_obj"] = short[obj_cols].sum(axis=1)
short = short.reset_index(drop=True)
short.head(8)

,gameid,position,side,teamname,result,teamkills,teamdeaths,dragons,elders,heralds,void_grubs,barons,atakhans,damagetochampions,totalgold,monsterkills,total_obj
0,LOLTMNT03_179647,team,Blue,IziDream,0,3,13,0.0,0.0,0.0,0.0,0.0,0.0,50143,42255,144,0.0
1,LOLTMNT03_179647,team,Red,Team Valiant,1,13,3,2.0,0.0,1.0,6.0,1.0,1.0,53681,53936,169,10.0
2,LOLTMNT06_96134,team,Blue,Esprit Shōnen,1,21,11,3.0,0.0,1.0,6.0,1.0,1.0,100112,64669,199,11.0
3,LOLTMNT06_96134,team,Red,Skillcamp,0,10,21,2.0,0.0,0.0,0.0,0.0,0.0,78010,50679,134,2.0
4,LOLTMNT06_95160,team,Blue,Karmine Corp Blue Stars,0,18,22,0.0,0.0,0.0,2.0,0.0,0.0,67870,51389,146,2.0
5,LOLTMNT06_95160,team,Red,Project Conquerors,1,22,18,4.0,0.0,1.0,4.0,1.0,1.0,86703,62965,180,10.0
6,LOLTMNT03_178705,team,Blue,Zerance,0,3,18,1.0,0.0,0.0,0.0,0.0,0.0,54173,40489,143,1.0
7,LOLTMNT03_178705,team,Red,Lille Esport,1,18,3,3.0,0.0,1.0,6.0,0.0,1.0,57648,51018,171,10.0


In [11]:
pivot = short.pivot(index="gameid", columns="side", values=["total_obj", "result"])

higher_obj_won = (
    (pivot["total_obj"]["Blue"] > pivot["total_obj"]["Red"]) & (pivot["result"]["Blue"] == 1)
) | (
    (pivot["total_obj"]["Red"] > pivot["total_obj"]["Blue"]) & (pivot["result"]["Red"] == 1)
)

win_percent = float(higher_obj_won.mean())
win_percent

0.7923008057296329

In [12]:
objectives = ["dragons", "barons", "elders", "heralds", "void_grubs", "atakhans"]

winrates = {}

for obj in objectives:
    more_col = f"more_{obj}"
    short[more_col] = (
        short[obj] >
        short.groupby("gameid")[obj].transform("mean")
    )
    winrates[obj] = short.loc[short[more_col], "result"].mean()

pd.Series(winrates)

dragons       0.830481
barons        0.904793
elders        0.869754
heralds       0.661583
void_grubs    0.599690
atakhans      0.796245
dtype: float64

In [13]:
rows = []

for obj in objectives:
    
    more = short[obj] > short.groupby("gameid")[obj].transform("mean")
    less = short[obj] < short.groupby("gameid")[obj].transform("mean")
    
    win_more = short.loc[more, "result"].mean()
    win_less = short.loc[less, "result"].mean()
    
    rows.append({
        "objective": obj,
        "winrate_more": win_more,
        "winrate_less": win_less,
        "winrate_diff": win_more - win_less
    })

objective_df = pd.DataFrame(rows).sort_values("winrate_diff", ascending=False)

objective_df

,objective,winrate_more,winrate_less,winrate_diff
1,barons,0.904793,0.095207,0.809586
2,elders,0.869754,0.130246,0.739508
0,dragons,0.830481,0.169519,0.660962
5,atakhans,0.796245,0.203755,0.592489
3,heralds,0.661583,0.338417,0.323166
4,void_grubs,0.599690,0.400310,0.199379


In [14]:

for m in objectives:
    short[f"{m}_diff"] = (
        2 * short[m] - short.groupby("gameid")[m].transform("sum")
    )

## Step 2: Data Cleaning and Exploratory Data Analysis

In [15]:
px.histogram(
    short,
    x=short["barons_diff"].abs(),
    title="Magnitude of Baron Advantage per Team",
    nbins=6,
    labels={"x": "Absolute Baron Difference"}
)

In [16]:
px.histogram(
    short,
    x=short["dragons_diff"].abs(),
    title="Magnitude of Dragon Advantage per Team",
    nbins=8,
    labels={"x": "Absolute Dragon Difference"}
)

## Step 3: Assessment of Missingness

In [17]:
# TODO

## Step 4: Hypothesis Testing

Null: The difference in winrate advantage between Baron and Dragon objectives is due to random chance.



Alternative: Baron objective advantage leads to a larger winrate difference than Dragon objective advantage.

Difference between:  (Winrate_more − Winrate_less) for Baron  (Winrate_more − Winrate_less) for Dragon

In [18]:
baron_more = short["barons_diff"] > 0
baron_less = short["barons_diff"] < 0

dragon_more = short["dragons_diff"] > 0
dragon_less = short["dragons_diff"] < 0

baron_gap = short.loc[baron_more, "result"].mean() - short.loc[baron_less, "result"].mean()
dragon_gap = short.loc[dragon_more, "result"].mean() - short.loc[dragon_less, "result"].mean()

observed_stat = baron_gap - dragon_gap
observed_stat

np.float64(0.14862396528386235)

In [31]:
n_permutations = 5000
perm_stats = []

for _ in range(n_permutations):

    shuffled_result = np.random.permutation(short["result"])

    baron_gap_perm = (
        shuffled_result[baron_more].mean() -
        shuffled_result[baron_less].mean()
    )

    dragon_gap_perm = (
        shuffled_result[dragon_more].mean() -
        shuffled_result[dragon_less].mean()
    )

    perm_stats.append(baron_gap_perm - dragon_gap_perm)

In [32]:
perm_stats = np.array(perm_stats)
p_value = (np.sum(perm_stats >= observed_stat) + 1) / (len(perm_stats) + 1)
p_value

np.float64(0.0001999600079984003)

In [33]:
px.histogram(
    perm_stats,
    nbins=40,
    title="Null Distribution of Test Statistic"
).add_vline(x=observed_stat, line_color="red")

## Step 5: Framing a Prediction Problem

In [19]:
# TODO

## Step 6: Baseline Model

In [36]:
X = short[["barons", "dragons"]]
y = short["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ("model", LogisticRegression())
])

pipe.fit(X_train, y_train)

preds = pipe.predict(X_test)

accuracy_score(y_test, preds)

0.8234709099950274

## Step 7: Final Model

In [21]:
# TODO

## Step 8: Fairness Analysis

In [22]:
# TODO